In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
traindf=pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
testdf=pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")
print("Training missing cells:", traindf.isna().sum().sum())
print("Test missing cells:", testdf.isna().sum().sum())

print("\nTraining missing values by column:")
print(traindf.isna().sum()[traindf.isna().sum() > 0])

print("\nTest missing values by column:")
print(testdf.isna().sum()[testdf.isna().sum() > 0])


Training missing cells: 292
Test missing cells: 130

Training missing values by column:
Arrival Delay in Minutes    292
dtype: int64

Test missing values by column:
Arrival Delay in Minutes    130
dtype: int64


In [3]:

X = traindf.drop(columns=["id", "satisfaction"])
y = traindf["satisfaction"]
X_test = testdf.drop(columns=["id"])

In [4]:
'''from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape, X_valid.shape)'''

'from sklearn.model_selection import train_test_split\n\nX_train, X_valid, y_train, y_valid = train_test_split(\n    X, y,\n    test_size=0.2,\n    random_state=42,\n    stratify=y\n)\nprint(X_train.shape, X_valid.shape)'

In [5]:
cat_cols = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print(cat_cols)

['Gender', 'Customer Type', 'Type of Travel', 'Class']


In [6]:
survey_cols = [
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness'
]

In [7]:
def feature_engineering_xgb(df):
    df['total_delay'] = (
        df['Departure Delay in Minutes']
        + df['Arrival Delay in Minutes']
    )

    df['delay_difference'] = (
        df['Arrival Delay in Minutes']
        - df['Departure Delay in Minutes']
    )

    df['is_departure_delayed'] = (
        df['Departure Delay in Minutes'] > 0
    ).astype(int)

    arrival = df['Arrival Delay in Minutes']

    df['is_arrival_delayed'] = (
        (arrival > 0)
        .where(arrival.notna())
        .astype('float')
    )
    return df

X_full_xgb = feature_engineering_xgb(X.copy())
X_test_xgb = feature_engineering_xgb(X_test.copy())

for col in cat_cols:
    X_full_xgb[col] = X_full_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

In [8]:

'''from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False
)

cat_model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100
)'''

'from catboost import CatBoostClassifier\n\ncat_model = CatBoostClassifier(\n    iterations=1000,\n    learning_rate=0.05,\n    depth=6,\n    eval_metric="AUC",\n    random_seed=42,\n    verbose=200,\n    allow_writing_files=False\n)\n\ncat_model.fit(\n    X_train, y_train,\n    cat_features=cat_cols,\n    eval_set=(X_valid, y_valid),\n    early_stopping_rounds=100\n)'

In [9]:
'''from sklearn.metrics import roc_auc_score

cat_probs = cat_model.predict_proba(X_valid)[:, 1]

score = roc_auc_score(y_valid, cat_probs)
print("Validation ROC-AUC:", score)'''

'from sklearn.metrics import roc_auc_score\n\ncat_probs = cat_model.predict_proba(X_valid)[:, 1]\n\nscore = roc_auc_score(y_valid, cat_probs)\nprint("Validation ROC-AUC:", score)'

In [10]:
'''import lightgbm as lgb
X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    categories = pd.concat([
        X_train_lgb[col],
        X_valid_lgb[col],
        X_test_lgb[col]
    ]).astype("category").cat.categories

    for df in [X_train_lgb, X_valid_lgb, X_test_lgb]:
        df[col] = pd.Categorical(df[col], categories=categories)'''

'import lightgbm as lgb\nX_train_lgb = X_train.copy()\nX_valid_lgb = X_valid.copy()\nX_test_lgb = X_test.copy()\nfor col in cat_cols:\n    categories = pd.concat([\n        X_train_lgb[col],\n        X_valid_lgb[col],\n        X_test_lgb[col]\n    ]).astype("category").cat.categories\n\n    for df in [X_train_lgb, X_valid_lgb, X_test_lgb]:\n        df[col] = pd.Categorical(df[col], categories=categories)'

In [11]:
'''print("Categorical columns:", cat_cols)
print("Training shape:", X_train_lgb.shape)
print("Validation shape:", X_valid_lgb.shape)'''

'print("Categorical columns:", cat_cols)\nprint("Training shape:", X_train_lgb.shape)\nprint("Validation shape:", X_valid_lgb.shape)'

In [12]:
'''lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_valid_lgb, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(200)
    ]
)'''

'lgb_model = lgb.LGBMClassifier(\n    n_estimators=1000,\n    learning_rate=0.05,\n    num_leaves=31,\n    random_state=42,\n    n_jobs=-1,\n    verbosity=-1\n)\n\nlgb_model.fit(\n    X_train_lgb,\n    y_train,\n    eval_set=[(X_valid_lgb, y_valid)],\n    eval_metric="auc",\n    callbacks=[\n        lgb.early_stopping(100),\n        lgb.log_evaluation(200)\n    ]\n)'

In [13]:
'''from sklearn.metrics import roc_auc_score
lgb_probs = lgb_model.predict_proba(X_valid_lgb)[:, 1]

lgb_auc = roc_auc_score(y_valid, lgb_probs)

print("LightGBM Validation ROC-AUC:", lgb_auc)'''

'from sklearn.metrics import roc_auc_score\nlgb_probs = lgb_model.predict_proba(X_valid_lgb)[:, 1]\n\nlgb_auc = roc_auc_score(y_valid, lgb_probs)\n\nprint("LightGBM Validation ROC-AUC:", lgb_auc)'

In [14]:
from xgboost import XGBClassifier
cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
final_model = XGBClassifier(
n_estimators=2500,
learning_rate=0.02724,
max_depth=7,
min_child_weight=1.48566,
subsample=0.80157,
colsample_bytree=0.80610,
reg_alpha=0.00001,
gamma=1.40412,
reg_lambda=0.7766532770119151,
tree_method="hist",
enable_categorical=True,
eval_metric="auc",
random_state=42,
n_jobs=-1 
)
final_model.fit(X_full_xgb,y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8061, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=1.40412, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.02724, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=1.48566, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [15]:
'''import optuna
def objective(trial):       
    params = {
        "n_estimators": 2500,
        "learning_rate": 0.02724,
        "max_depth": 7,
        "min_child_weight": 1.48566,
        "subsample": 0.80157,
        "colsample_bytree": 0.80610,
        "gamma": 1.40412,
        "reg_alpha": 0.00001,
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.01, 20, log=True
        ),
        "tree_method": "hist",
        "enable_categorical": True,
        "eval_metric": "auc",
        "early_stopping_rounds": 50,
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0
        }
    model = XGBClassifier(**params)

    model.fit(
        X_train_xgb,
        y_train,
        eval_set=[(X_valid_xgb, y_valid)],
        verbose=False
    )

    return float(model.best_score)


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=30)

print("Best validation AUC:", study.best_value)
print("Best parameters:", study.best_params)'''

'import optuna\ndef objective(trial):       \n    params = {\n        "n_estimators": 2500,\n        "learning_rate": 0.02724,\n        "max_depth": 7,\n        "min_child_weight": 1.48566,\n        "subsample": 0.80157,\n        "colsample_bytree": 0.80610,\n        "gamma": 1.40412,\n        "reg_alpha": 0.00001,\n        "reg_lambda": trial.suggest_float(\n            "reg_lambda", 0.01, 20, log=True\n        ),\n        "tree_method": "hist",\n        "enable_categorical": True,\n        "eval_metric": "auc",\n        "early_stopping_rounds": 50,\n        "random_state": 42,\n        "n_jobs": -1,\n        "verbosity": 0\n        }\n    model = XGBClassifier(**params)\n\n    model.fit(\n        X_train_xgb,\n        y_train,\n        eval_set=[(X_valid_xgb, y_valid)],\n        verbose=False\n    )\n\n    return float(model.best_score)\n\n\nstudy = optuna.create_study(direction="maximize")\n\nstudy.optimize(objective, n_trials=30)\n\nprint("Best validation AUC:", study.best_value)\n

In [16]:
'''ensemble_probs=(cat_probs+lgb_probs+xgb_probs)/3.0;
ensemble_auc = roc_auc_score(y_valid, ensemble_probs)
print(ensemble_auc)'''

'ensemble_probs=(cat_probs+lgb_probs+xgb_probs)/3.0;\nensemble_auc = roc_auc_score(y_valid, ensemble_probs)\nprint(ensemble_auc)'

In [17]:
'''w_cat=0
w_lgb=0
w_xg=0
def objective(trial):
    raw_cat = trial.suggest_float("raw_cat", 0, 1)
    raw_lgb = trial.suggest_float("raw_lgb", 0, 1)
    raw_xgb = trial.suggest_float("raw_xgb", 0, 1)

    total = raw_cat + raw_lgb + raw_xgb

    w_cat = raw_cat / total
    w_lgb = raw_lgb / total
    w_xgb = raw_xgb / total

    ensemble_probs = (
        w_cat * cat_probs
        + w_lgb * lgb_probs
        + w_xgb * xgb_probs
    )

    return roc_auc_score(y_valid, ensemble_probs)'''

'w_cat=0\nw_lgb=0\nw_xg=0\ndef objective(trial):\n    raw_cat = trial.suggest_float("raw_cat", 0, 1)\n    raw_lgb = trial.suggest_float("raw_lgb", 0, 1)\n    raw_xgb = trial.suggest_float("raw_xgb", 0, 1)\n\n    total = raw_cat + raw_lgb + raw_xgb\n\n    w_cat = raw_cat / total\n    w_lgb = raw_lgb / total\n    w_xgb = raw_xgb / total\n\n    ensemble_probs = (\n        w_cat * cat_probs\n        + w_lgb * lgb_probs\n        + w_xgb * xgb_probs\n    )\n\n    return roc_auc_score(y_valid, ensemble_probs)'

In [18]:
'''import optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print("Best ROC-AUC:", study.best_value)
print("Best parameters:", study.best_params)'''

'import optuna\nstudy = optuna.create_study(direction="maximize")\nstudy.optimize(objective, n_trials=100)\n\nprint("Best ROC-AUC:", study.best_value)\nprint("Best parameters:", study.best_params)'

In [19]:

test_probs = final_model.predict_proba(X_test_xgb)[:, 1]
submission = pd.DataFrame({
    "id": testdf["id"],
    "satisfaction": test_probs
})
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())


       id  satisfaction
0  699635      0.233505
1  699636      0.898762
2  699637      0.897046
3  699638      0.078721
4  699639      0.060043
